# Fourier Series and the DFT/FFT

Any reasonable periodic signal — a square wave, a sawtooth, a recorded vowel — is a sum of rotating arrows $c_n e^{inx}$ spinning at whole-number speeds. Finding the arrows is a projection: the functions $e^{inx}$ are **orthogonal**, so each coefficient $c_n$ is an inner product. Sampling the signal turns the integral into a finite sum over roots of unity, the **discrete Fourier transform**, which the FFT computes in $O(N\log N)$ operations and which sits inside every audio codec, image format and spectrum analyser.

This notebook develops the complex Fourier series, watches it converge (and overshoot, the Gibbs phenomenon), moves from sampling to the DFT and `np.fft`, finds hidden tones in a noisy signal, and proves the **convolution theorem** that makes filtering a multiplication.

This is notebook 03 of module 07 (**Applications**) in the **Learning Complex Numbers** series.

## 1. Introduction

In 1807 Joseph Fourier claimed that *every* periodic function, even one with jumps, could be written as a sum of sines and cosines. The claim was controversial (Lagrange objected) and it took most of the 19th century to make it precise, but it reshaped mathematics and physics.

In real form a Fourier series is a bookkeeping exercise with two families, $\cos nx$ and $\sin nx$, and two coefficient formulas. In complex form there is **one** family, $e^{inx}$ for $n \in \mathbb{Z}$, **one** coefficient formula, and a clear geometric meaning: $c_n e^{inx}$ is an arrow of length $|c_n|$ that turns $n$ times per period, starting at angle $\arg c_n$. Negative $n$ turns clockwise. For a real signal the arrows come in conjugate pairs whose imaginary parts cancel.

Sampled at $N$ points, the same idea becomes linear algebra in $\mathbb{C}^N$: the samples of $e^{inx}$ are powers of an $N$-th root of unity, and the DFT is a change of basis into these vectors.

**Prerequisites**

- Euler's formula — [Euler's formula](../01_polar_form_and_euler/03_eulers_formula.ipynb).
- Roots of unity — [Roots of unity and n-th roots](../01_polar_form_and_euler/05_roots_of_unity_and_nth_roots.ipynb).
- Complex inner products and orthogonality — [Complex vectors and inner products](../04_complex_linear_algebra/03_complex_vectors_and_inner_products.ipynb).
- The DFT as a matrix — [The DFT matrix](../04_complex_linear_algebra/05_the_dft_matrix.ipynb).
- Discrete convolution — [Convolution](../../../LearningNeuralNetworks/prerequisites/09_convolution.ipynb) (LearningNeuralNetworks).

After completing this notebook you will be able to:

- Compute complex Fourier coefficients $c_n$ by hand and numerically, and relate them to sine/cosine coefficients.
- Explain orthogonality of $e^{inx}$ and use it to derive the coefficient formula.
- Describe convergence of partial sums, the Gibbs overshoot, and how smoothness controls the decay of $c_n$.
- Go from samples to the DFT, use `np.fft` with correct frequency axes, and recognise aliasing.
- State and use the convolution theorem.

## 2. Theory

### 2.1 Orthogonality of $e^{inx}$

On functions of period $2\pi$ use the inner product $\langle f, g\rangle = \frac{1}{2\pi}\int_{-\pi}^{\pi} f(x)\overline{g(x)}\,dx$. For integers $m, n$:

$$
\langle e^{imx}, e^{inx}\rangle = \frac{1}{2\pi}\int_{-\pi}^{\pi} e^{i(m - n)x}\,dx = \begin{cases} 1 & m = n, \\[2pt] \dfrac{e^{i(m-n)\pi} - e^{-i(m-n)\pi}}{2\pi i(m - n)} = \dfrac{\sin((m-n)\pi)}{\pi(m-n)} = 0 & m \neq n. \end{cases}
$$

Geometrically: for $k = m - n \ne 0$ the point $e^{ikx}$ runs $k$ full times around the unit circle as $x$ goes from $-\pi$ to $\pi$, so its average position is the centre, $0$.

### 2.2 The complex Fourier series

If $f(x) = \sum_{n=-\infty}^{\infty} c_n e^{inx}$, take the inner product with $e^{imx}$: orthogonality kills every term except $n = m$, giving

$$
\boxed{\,c_n = \langle f, e^{inx}\rangle = \frac{1}{2\pi}\int_{-\pi}^{\pi} f(x)\,e^{-inx}\,dx\,}
$$

This is the formula for the coordinates of a vector in an orthonormal basis, applied to functions. The **partial sum** $S_N f(x) = \sum_{|n| \le N} c_n e^{inx}$ is the best approximation of $f$ (in mean square) by such sums.

**Real signals.** If $f$ is real then $c_{-n} = \overline{c_n}$, so $c_n e^{inx} + c_{-n}e^{-inx} = 2\,\mathrm{Re}(c_n e^{inx})$ is real. With the real series $f = \frac{a_0}{2} + \sum_{n\ge1}(a_n\cos nx + b_n\sin nx)$:

$$
c_n = \frac{a_n - i\,b_n}{2}, \qquad c_{-n} = \frac{a_n + i\,b_n}{2}, \qquad c_0 = \frac{a_0}{2}.
$$

**Parseval's identity** (Pythagoras for the orthonormal basis): $\frac{1}{2\pi}\int_{-\pi}^{\pi}|f|^2\,dx = \sum_n |c_n|^2$.

### 2.3 Two standard examples

| Signal on $(-\pi, \pi)$, extended periodically | $c_n$ ($n \neq 0$) | real form |
|---|---|---|
| square wave $\mathrm{sign}(x)$ | $\dfrac{1 - (-1)^n}{i\pi n}$: $\ -\dfrac{2i}{\pi n}$ for odd $n$, $0$ for even | $\dfrac{4}{\pi}\left(\sin x + \dfrac{\sin 3x}{3} + \dfrac{\sin 5x}{5} + \cdots\right)$ |
| sawtooth $x$ | $\dfrac{i(-1)^n}{n}$ | $2\left(\sin x - \dfrac{\sin 2x}{2} + \dfrac{\sin 3x}{3} - \cdots\right)$ |

Both have $c_0 = 0$ and coefficients decaying like $1/n$.

### 2.4 Convergence, smoothness and the Gibbs phenomenon

- **Smoothness controls decay.** A jump gives $|c_n| \sim 1/n$; a continuous function with corners (triangle wave $|x|$) gives $1/n^2$; each further derivative buys another power of $n$. Integration by parts shows why: $c_n(f') = in\,c_n(f)$.
- **At a jump** the partial sums converge to the midpoint of the jump.
- **Gibbs phenomenon.** Near a jump, $S_N f$ overshoots by about $9\%$ of the jump height, and the overshoot does **not** shrink as $N \to \infty$ — it only gets narrower. For the square wave the peak of $S_N$ tends to $\frac{2}{\pi}\mathrm{Si}(\pi) \approx 1.1790$.

### 2.5 Sampling: from the integral to the DFT

Sample $f$ at $N$ equally spaced points $x_k = 2\pi k/N$, $k = 0, \dots, N - 1$ (integrate over $[0, 2\pi)$, which is also a full period). The Riemann sum for $c_n$ is

$$
c_n \approx \frac{1}{N}\sum_{k=0}^{N-1} f(x_k)\,e^{-inx_k} = \frac{1}{N}\sum_{k=0}^{N-1} f_k\,\omega^{-nk}, \qquad \omega = e^{2\pi i/N}.
$$

This is the **discrete Fourier transform**:

$$
X_n = \sum_{k=0}^{N-1} x_k\,e^{-2\pi i nk/N}, \qquad x_k = \frac{1}{N}\sum_{n=0}^{N-1} X_n\,e^{2\pi i nk/N}.
$$

The inverse works because the sampled exponentials are orthogonal in $\mathbb{C}^N$: $\sum_{k=0}^{N-1}\omega^{(m-n)k} = N$ if $m \equiv n \pmod N$ and $0$ otherwise (a sum of all the $N$-th roots of unity, or a geometric series).

Key facts:

- **Periodicity / aliasing.** $\omega^{(n+N)k} = \omega^{nk}$, so $X_{n+N} = X_n$: on $N$ samples, frequency $n$ and $n + N$ are indistinguishable. Index $N - n$ represents frequency $-n$. With sampling rate $f_s$, tones above the **Nyquist frequency** $f_s/2$ fold back to lower frequencies.
- **Real input.** $X_{N-n} = \overline{X_n}$, so the second half of the spectrum mirrors the first.
- **Amplitude.** A tone $A\cos(2\pi f t)$ with an integer number of periods in the window gives $|X_n| = AN/2$ at $n = \pm f$; the one-sided amplitude estimate is $2|X_n|/N$.
- **FFT.** Evaluating the sums directly costs $O(N^2)$. The fast Fourier transform (Cooley–Tukey, 1965, anticipated by Gauss) splits even and odd samples recursively and costs $O(N\log N)$. `np.fft.fft` uses the sign and scaling convention above; `np.fft.fftfreq(N, d)` gives the frequency of each bin.

### 2.6 The convolution theorem

The **circular convolution** of two length-$N$ sequences is $(x * h)_k = \sum_{m=0}^{N-1} x_m\,h_{(k - m) \bmod N}$. Then

$$
\mathrm{DFT}(x * h)_n = X_n\,H_n.
$$

*Proof.* $\sum_k \sum_m x_m h_{k-m}\,\omega^{-nk} = \sum_m x_m\omega^{-nm}\sum_k h_{k-m}\,\omega^{-n(k-m)} = X_n H_n$, since $k - m$ runs over all residues mod $N$.

Convolution — sliding a kernel along a signal, as in a moving average or a convolutional neural network — becomes **pointwise multiplication** of spectra. Each $X_n$ is scaled and rotated by the complex number $H_n$: the filter's gain and phase shift at that frequency. Zero-padding both sequences to length at least $\mathrm{len}(x) + \mathrm{len}(h) - 1$ makes circular convolution equal ordinary (linear) convolution. (The "convolution" layer of a neural network actually computes a cross-correlation $\sum_m x_{k+m}h_m$, whose DFT is $X_n\overline{H_n}$ for real $h$.)

## 3. Worked Examples

### Example 1 — an orthogonality integral

$\langle e^{3ix}, e^{ix}\rangle = \frac{1}{2\pi}\int_{-\pi}^{\pi}e^{2ix}\,dx = \frac{1}{2\pi}\left[\frac{e^{2ix}}{2i}\right]_{-\pi}^{\pi} = \frac{e^{2\pi i} - e^{-2\pi i}}{4\pi i} = 0$.

### Example 2 — the square wave

With $f = -1$ on $(-\pi, 0)$ and $+1$ on $(0, \pi)$:

$$
c_n = \frac{1}{2\pi}\left(\int_0^{\pi} e^{-inx}dx - \int_{-\pi}^{0} e^{-inx}dx\right) = \frac{1}{2\pi}\left(\frac{1 - (-1)^n}{in} - \frac{(-1)^n - 1}{in}\right) = \frac{1 - (-1)^n}{i\pi n}.
$$

For odd $n$ this is $\frac{2}{i\pi n} = -\frac{2i}{\pi n}$; for even $n \ne 0$ it is $0$; and $c_0 = 0$ (the average). Pairing $\pm n$:

$$
c_n e^{inx} + c_{-n}e^{-inx} = -\frac{2i}{\pi n}\left(e^{inx} - e^{-inx}\right) = -\frac{2i}{\pi n}\cdot 2i\sin nx = \frac{4}{\pi n}\sin nx.
$$

Each arrow $c_n$ points straight down ($-90°$) for $n > 0$: the square wave is odd, built from sines.

### Example 3 — the sawtooth

For $f(x) = x$, integrate by parts with $u = x$, $dv = e^{-inx}dx$:

$$
c_n = \frac{1}{2\pi}\left(\left[\frac{x e^{-inx}}{-in}\right]_{-\pi}^{\pi} + \frac{1}{in}\int_{-\pi}^{\pi}e^{-inx}dx\right) = \frac{1}{2\pi}\cdot\frac{\pi(-1)^n + \pi(-1)^n}{-in} = \frac{i(-1)^n}{n}.
$$

Since $a_n = 0$, $c_n = -\frac{i b_n}{2}$ gives $b_n = 2i\,c_n = -\frac{2(-1)^n}{n} = \frac{2(-1)^{n+1}}{n}$, matching the table.

### Example 4 — a 4-point DFT by hand

For $x = (1, 2, 3, 4)$ the twiddle factor is $\omega^{-1} = e^{-2\pi i/4} = -i$, so $X_n = \sum_k x_k(-i)^{nk}$:

- $X_0 = 1 + 2 + 3 + 4 = 10$,
- $X_1 = 1 + 2(-i) + 3(-1) + 4(i) = -2 + 2i$,
- $X_2 = 1 - 2 + 3 - 4 = -2$,
- $X_3 = 1 + 2(i) + 3(-1) + 4(-i) = -2 - 2i = \overline{X_1}$ (real input).

### Example 5 — a pure tone

$x_k = \cos(2\pi\cdot 3k/16)$, $N = 16$. Writing $\cos\theta = \frac{e^{i\theta} + e^{-i\theta}}{2}$, the signal is $\frac12(\omega^{3k} + \omega^{-3k})$, i.e. half of basis vector $3$ plus half of basis vector $-3 \equiv 13$. By orthogonality $X_3 = X_{13} = \frac{16}{2} = 8$ and all other $X_n = 0$. The amplitude estimate $2|X_3|/N = 1$ recovers the amplitude.

### Example 6 — the convolution theorem on 4 points

$x = (1, 2, 0, 0)$, $h = (1, 1, 0, 0)$ (a two-tap sum). Circularly, $y = x * h = (1, 3, 2, 0)$.

DFTs: $X = (3,\ 1 - 2i,\ -1,\ 1 + 2i)$ and $H = (2,\ 1 - i,\ 0,\ 1 + i)$. Products: $(6,\ (1 - 2i)(1 - i),\ 0,\ (1 + 2i)(1 + i)) = (6,\ -1 - 3i,\ 0,\ -1 + 3i)$.

Directly: $Y_1 = 1 + 3(-i) + 2(-1) + 0 = -1 - 3i$. ✓ Note $H_2 = 0$: the kernel $(1, 1)$ completely removes the fastest frequency (the alternating pattern $+, -, +, -$).

## 4. Python Examples

### 4.1 Coefficients by numerical integration

We compute $c_n$ for the square wave and sawtooth with the trapezoidal rule and compare with the formulas of section 2.3.

In [ ]:
import numpy as np

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

x = np.linspace(-np.pi, np.pi, 200001)
signals = {
    "square": (np.sign(x), lambda n: (1 - (-1) ** n) / (1j * np.pi * n)),
    "sawtooth": (x, lambda n: 1j * (-1) ** n / n),
}


def fourier_coefficient(values, n):
    return np.trapezoid(values * np.exp(-1j * n * x), x) / (2 * np.pi)


for name, (values, formula) in signals.items():
    print(name)
    for n in [1, 2, 3, -3]:
        numeric = fourier_coefficient(values, n)
        print(f"  c_{n:+d}: numeric {numeric.real:+.5f}{numeric.imag:+.5f}j   formula {formula(n).real:+.5f}{formula(n).imag:+.5f}j")

Sympy gives the same coefficients exactly, for a general integer $n$:

In [ ]:
import sympy as sp

x = sp.symbols("x", real=True)
n = sp.symbols("n", integer=True, nonzero=True)

c_square = (sp.integrate(sp.exp(-sp.I * n * x), (x, 0, sp.pi)) - sp.integrate(sp.exp(-sp.I * n * x), (x, -sp.pi, 0))) / (2 * sp.pi)
c_saw = sp.integrate(x * sp.exp(-sp.I * n * x), (x, -sp.pi, sp.pi)) / (2 * sp.pi)

# rewrite(sp.cos) turns exp(i pi n) into cos(pi n) = (-1)^n
print("square   c_n =", sp.simplify(c_square.rewrite(sp.cos)))
print("sawtooth c_n =", sp.simplify(c_saw.rewrite(sp.cos)))
print("difference from the table formulas:",
      sp.simplify(c_square - (1 - (-1) ** n) / (sp.I * sp.pi * n)), sp.simplify(c_saw - sp.I * (-1) ** n / n))
print("square c_1, c_2, c_3 =", [sp.simplify(c_square.subs(n, k)) for k in (1, 2, 3)])

### 4.2 A DFT written from the definition vs `np.fft.fft`

The direct formula is one matrix–vector product with the $N \times N$ matrix $W_{nk} = \omega^{-nk}$ (notebook 04/05). It matches `np.fft.fft` to rounding error.

In [ ]:
import time

import numpy as np


def dft(signal):
    N = len(signal)
    k = np.arange(N)
    W = np.exp(-2j * np.pi * np.outer(k, k) / N)
    return W @ signal


print("Ex 4:", np.round(dft(np.array([1, 2, 3, 4], dtype=complex)), 10))
print("np.fft.fft:", np.fft.fft([1, 2, 3, 4]))

N = 16
tone = np.cos(2 * np.pi * 3 * np.arange(N) / N)
print("Ex 5 nonzero bins:", {n: np.round(v, 10) for n, v in enumerate(np.fft.fft(tone)) if abs(v) > 1e-9})

rng = np.random.default_rng(0)
signal = rng.standard_normal(2048)
start = time.perf_counter()
slow = dft(signal)
middle = time.perf_counter()
fast = np.fft.fft(signal)
end = time.perf_counter()
print(f"N = 2048: max difference {np.max(np.abs(slow - fast)):.1e};  direct {1000 * (middle - start):.1f} ms, FFT {1000 * (end - middle):.3f} ms")
print("inverse recovers the signal:", np.allclose(np.fft.ifft(fast).real, signal))

### 4.3 Sampled DFT approximates the Fourier coefficients

Sampling the square wave at $N = 64$ points, $X_n/N$ approximates $c_n$ for small $|n|$ (negative $n$ live at index $N + n$). The approximation degrades as $|n|$ approaches $N/2$, where aliasing mixes in the coefficients $c_{n \pm N}, c_{n \pm 2N}, \dots$.

In [ ]:
import numpy as np

N = 64
x_samples = 2 * np.pi * np.arange(N) / N
square = np.where(x_samples < np.pi, 1.0, -1.0)
square[0] = 0.0
square[N // 2] = 0.0        # midpoints of the two jumps
X = np.fft.fft(square)

for n in [1, 3, 5, -1, 15, 25]:
    exact = (1 - (-1) ** n) / (1j * np.pi * n)
    approx = X[n % N] / N
    print(f"n = {n:+3d}:  X_n/N = {approx.imag:+.5f}j   exact c_n = {exact.imag:+.5f}j   error {abs(approx - exact):.1e}")

### 4.4 Finding tones in a noisy signal

One second of a signal sampled at $1000$ Hz contains a $50$ Hz tone of amplitude $1$ and a $120$ Hz tone of amplitude $0.5$, buried in Gaussian noise of standard deviation $1$. In the time domain the tones are hard to make out; in the spectrum they stand out, because the noise spreads its energy over all $500$ bins while each tone concentrates in one.

In [ ]:
import numpy as np

sample_rate = 1000
duration = 1.0
t = np.arange(0, duration, 1 / sample_rate)
clean = 1.0 * np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
rng = np.random.default_rng(42)
noisy = clean + rng.normal(0, 1.0, size=t.size)

N = t.size
spectrum = np.fft.fft(noisy)
frequencies = np.fft.fftfreq(N, d=1 / sample_rate)
positive = frequencies >= 0
amplitude = 2 * np.abs(spectrum[positive]) / N

top_bins = np.argsort(amplitude)[::-1][:4]
for b in top_bins:
    print(f"{frequencies[positive][b]:6.1f} Hz  amplitude estimate {amplitude[b]:.3f}")
print("median noise amplitude per bin:", round(float(np.median(amplitude)), 3))

# Denoise: keep only bins well above the noise floor, then invert
threshold = 0.3 * N / 2
filtered_spectrum = np.where(np.abs(spectrum) > threshold, spectrum, 0)
denoised = np.fft.ifft(filtered_spectrum).real
print("RMS error noisy   :", round(float(np.sqrt(np.mean((noisy - clean) ** 2))), 3))
print("RMS error denoised:", round(float(np.sqrt(np.mean((denoised - clean) ** 2))), 3))

### 4.5 Aliasing

Sampled at $100$ Hz, a $70$ Hz cosine produces *exactly the same samples* as a $30$ Hz cosine, because $70 = 100 - 30$ and $\cos(2\pi\cdot 70k/100) = \cos(2\pi k - 2\pi\cdot 30k/100)$.

In [ ]:
import numpy as np

sample_rate = 100
k = np.arange(100)
fast = np.cos(2 * np.pi * 70 * k / sample_rate)
slow = np.cos(2 * np.pi * 30 * k / sample_rate)
print("identical samples:", np.allclose(fast, slow))
peak = np.argmax(np.abs(np.fft.fft(fast))[: len(k) // 2])
print("FFT peak of the 70 Hz tone at", np.fft.fftfreq(len(k), 1 / sample_rate)[peak], "Hz")

### 4.6 The convolution theorem in code

Circular convolution computed by the definition equals `ifft(fft(x) * fft(h))`; with zero padding it also equals `np.convolve`.

In [ ]:
import numpy as np


def circular_convolution(x, h):
    N = len(x)
    return np.array([sum(x[m] * h[(k - m) % N] for m in range(N)) for k in range(N)])


x = np.array([1.0, 2.0, 0.0, 0.0])
h = np.array([1.0, 1.0, 0.0, 0.0])
print("Ex 6 by definition:", circular_convolution(x, h))
print("Ex 6 via FFT      :", np.round(np.fft.ifft(np.fft.fft(x) * np.fft.fft(h)).real, 12))
print("X * H             :", np.round(np.fft.fft(x) * np.fft.fft(h), 12))

rng = np.random.default_rng(1)
signal = rng.standard_normal(50)
kernel = np.ones(5) / 5
size = len(signal) + len(kernel) - 1
via_fft = np.fft.ifft(np.fft.fft(signal, size) * np.fft.fft(kernel, size)).real
print("zero-padded FFT convolution == np.convolve:", np.allclose(via_fft, np.convolve(signal, kernel)))

## 5. Visualizations

### 5.1 Orthogonality as a Gram matrix

The matrix of inner products $\langle e^{imx}, e^{inx}\rangle$, here computed from $N = 32$ samples, is the identity: every basis function is orthogonal to every other one and has unit length. On the right, the samples of $e^{2ix}$ as points on the unit circle: they visit the circle twice, evenly, so their average is $0$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 32
x = 2 * np.pi * np.arange(N) / N
modes = np.arange(-4, 5)
basis = np.exp(1j * np.outer(modes, x))
gram = basis @ basis.conj().T / N

fig, (ax_gram, ax_circle) = plt.subplots(1, 2, figsize=(12, 5.5))
image = ax_gram.imshow(np.abs(gram), cmap="viridis", vmin=0, vmax=1)
ax_gram.set_xticks(range(len(modes)))
ax_gram.set_xticklabels(modes)
ax_gram.set_yticks(range(len(modes)))
ax_gram.set_yticklabels(modes)
ax_gram.set_xlabel("n")
ax_gram.set_ylabel("m")
ax_gram.set_title("$|\\langle e^{imx}, e^{inx} \\rangle|$ from 32 samples")
fig.colorbar(image, ax=ax_gram, fraction=0.046)

points = np.exp(2j * x)
ax_circle.plot(np.cos(np.linspace(0, 2 * np.pi, 200)), np.sin(np.linspace(0, 2 * np.pi, 200)), color="lightgray")
ax_circle.scatter(points.real, points.imag, c=np.arange(N), cmap="plasma", s=60, zorder=3)
ax_circle.plot(points.mean().real, points.mean().imag, "k*", markersize=15, label="average = 0")
ax_circle.axhline(0, color="black", linewidth=0.8)
ax_circle.axvline(0, color="black", linewidth=0.8)
ax_circle.set_aspect("equal")
ax_circle.set_xlim(-1.4, 1.4)
ax_circle.set_ylim(-1.4, 1.4)
ax_circle.set_xlabel("Re")
ax_circle.set_ylabel("Im")
ax_circle.set_title("Samples of $e^{2ix}$ (colour = sample index)")
ax_circle.legend(loc="upper right")
ax_circle.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Partial sums of the square wave and the Gibbs overshoot

Adding more terms sharpens the edges, but next to each jump the partial sum overshoots to about $1.18$ — no matter how many terms. The right panel zooms in near the jump at $x = 0$: the overshoot moves closer to the jump as $N$ grows, but its height stays put.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-np.pi, np.pi, 4001)


def square_partial_sum(x, N):
    total = np.zeros_like(x, dtype=complex)
    for n in range(-N, N + 1):
        if n % 2 != 0:
            total += (2 / (1j * np.pi * n)) * np.exp(1j * n * x)
    return total.real


fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(13, 5))
ax_full.plot(x, np.sign(x), color="black", linewidth=2, label="square wave")
for N in [1, 3, 9, 31]:
    ax_full.plot(x, square_partial_sum(x, N), label=f"$S_{{{N}}}$")
for N in [9, 31, 101]:
    ax_zoom.plot(x, square_partial_sum(x, N), label=f"$S_{{{N}}}$")
ax_zoom.axhline(1, color="black", linewidth=1)
ax_zoom.axhline(1.1790, color="gray", linestyle="--", label="1.179 (Gibbs limit)")
ax_zoom.set_xlim(0, 0.8)
ax_zoom.set_ylim(0.6, 1.25)
for ax, title in [(ax_full, "Partial sums $S_N$ of the square wave"), (ax_zoom, "Zoom next to the jump at $x = 0$")]:
    ax.set_xlabel("x")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

for N in [9, 31, 101, 301]:
    print(f"N = {N:3d}: max S_N = {square_partial_sum(np.linspace(0, 0.5, 20001), N).max():.4f}")

### 5.3 A Fourier series is a chain of rotating arrows

At a fixed $x$, the partial sum $S_N f(x)$ is a chain of arrows $c_n e^{inx}$ laid head to tail. Here $f$ is the square wave, $x = 1$ and $N = 15$, ordered $c_0, c_1, c_{-1}, c_3, c_{-3}, \dots$. Each conjugate pair is mirror-symmetric, so the chain wanders off the real axis and comes back: the endpoint is real, close to $f(1) = 1$. The big "V" is the pair $n = \pm 1$; the small arrows near the tip are the higher harmonics, correcting the value. As $x$ changes, every arrow turns at its own speed $n$ — the "epicycles" of Ptolemaic astronomy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_value, N = 1.0, 15
order = [0] + [s * n for n in range(1, N + 1, 2) for s in (1, -1)]
terms = [((1 - (-1) ** n) / (1j * np.pi * n) if n != 0 else 0j) * np.exp(1j * n * x_value) for n in order]

fig, ax = plt.subplots(figsize=(9, 6))
position = 0j
colors = plt.cm.viridis(np.linspace(0, 0.9, len(terms)))
for n, term, color in zip(order, terms, colors):
    if abs(term) < 1e-12:
        continue
    end = position + term
    ax.annotate("", xy=(end.real, end.imag), xytext=(position.real, position.imag),
                arrowprops=dict(arrowstyle="->", color=color, linewidth=1.8))
    if abs(n) == 1:
        middle = position + term / 2
        ax.text(middle.real, middle.imag - 0.05, f"n = {n}", fontsize=10, color=color, ha="center")
    position = end

ax.plot(position.real, position.imag, "r*", markersize=15, label=f"$S_{{{N}}}(1)$ = {position.real:.3f} (Im = {abs(position.imag):.1f})")
ax.plot(1, 0, "ko", label="f(1) = 1")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_aspect("equal")
ax.set_xlim(-0.1, 1.3)
ax.set_ylim(-0.5, 0.35)
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.set_title("Square wave at x = 1: arrows $c_n e^{inx}$ head to tail")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.show()

### 5.4 Smoothness and coefficient decay

On log-log axes, $|c_n|$ for the square wave and sawtooth (both with jumps) fall with slope $-1$, while the continuous triangle wave $|x|$, with $c_n = -\frac{2}{\pi n^2}$ for odd $n$, falls with slope $-2$. That is why the triangle wave's partial sums converge so much faster and show no Gibbs overshoot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n = np.arange(1, 200)
odd = n[n % 2 == 1]

plt.figure(figsize=(8, 5))
plt.loglog(odd, 2 / (np.pi * odd), "o", markersize=4, label="square wave: $2/(\\pi n)$, odd n")
plt.loglog(n, 1 / n, ".", markersize=4, label="sawtooth: $1/n$")
plt.loglog(odd, 2 / (np.pi * odd**2), "s", markersize=4, label="triangle $|x|$: $2/(\\pi n^2)$, odd n")
plt.xlabel("n")
plt.ylabel("$|c_n|$")
plt.title("Jumps give $1/n$ decay, corners give $1/n^2$")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

### 5.5 The noisy signal: time domain vs frequency domain

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_rate = 1000
t = np.arange(0, 1.0, 1 / sample_rate)
clean = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
noisy = clean + np.random.default_rng(42).normal(0, 1.0, size=t.size)

N = t.size
spectrum = np.fft.fft(noisy)
frequencies = np.fft.fftfreq(N, d=1 / sample_rate)
positive = frequencies >= 0
denoised = np.fft.ifft(np.where(np.abs(spectrum) > 0.3 * N / 2, spectrum, 0)).real

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
axes[0].plot(t[:200] * 1000, noisy[:200], color="lightgray", label="noisy")
axes[0].plot(t[:200] * 1000, clean[:200], color="tab:blue", label="clean (hidden)")
axes[0].set_xlabel("t (ms)")
axes[0].set_title("Time domain: the tones are hard to see in the noise")
axes[0].legend(loc="upper right")

axes[1].plot(frequencies[positive], 2 * np.abs(spectrum[positive]) / N, color="tab:purple")
axes[1].axhline(0.3, color="tab:red", linestyle="--", label="threshold")
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_ylabel("amplitude $2|X_n|/N$")
axes[1].set_title("Frequency domain: peaks at 50 Hz and 120 Hz")
axes[1].legend()

axes[2].plot(t[:200] * 1000, clean[:200], color="tab:blue", linewidth=3, alpha=0.4, label="clean")
axes[2].plot(t[:200] * 1000, denoised[:200], color="tab:orange", label="reconstructed from 4 bins")
axes[2].set_xlabel("t (ms)")
axes[2].set_title("Keeping only the strong bins and inverting")
axes[2].legend(loc="upper right")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.6 Filtering is multiplication: a moving average in both domains

A 9-point moving average is a convolution with the kernel $h = (\tfrac19, \dots, \tfrac19)$. Its DFT $H_n$ (left, in the complex plane, and centre, as a gain curve) is close to $1$ at low frequencies and small at high frequencies, so multiplying spectra by $H$ smooths the signal (right). The points $H_n$ trace a curve in the complex plane: each frequency is scaled by $|H_n|$ and rotated by $\arg H_n$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N, width = 256, 9
kernel = np.zeros(N)
kernel[:width] = 1 / width
H = np.fft.fft(kernel)
frequencies = np.fft.fftfreq(N)

rng = np.random.default_rng(3)
k = np.arange(N)
signal = np.sin(2 * np.pi * 3 * k / N) + 0.4 * rng.standard_normal(N)
smoothed = np.fft.ifft(np.fft.fft(signal) * H).real

fig, (ax_plane, ax_gain, ax_signal) = plt.subplots(1, 3, figsize=(15, 4.8))
ax_plane.plot(H.real, H.imag, ".", color="tab:purple", markersize=4)
ax_plane.plot(1, 0, "ko", label="$H_0 = 1$")
ax_plane.axhline(0, color="black", linewidth=0.8)
ax_plane.axvline(0, color="black", linewidth=0.8)
ax_plane.set_aspect("equal")
ax_plane.set_xlabel("Re")
ax_plane.set_ylabel("Im")
ax_plane.set_title("$H_n$ in the complex plane")
ax_plane.legend()

order = np.argsort(frequencies)
ax_gain.plot(frequencies[order], np.abs(H[order]), color="tab:purple")
ax_gain.set_xlabel("frequency (cycles per sample)")
ax_gain.set_ylabel("$|H_n|$")
ax_gain.set_title("Gain: a low-pass filter")

ax_signal.plot(k, signal, color="lightgray", label="signal")
ax_signal.plot(k, smoothed, color="tab:orange", label="ifft(X H)")
ax_signal.set_xlabel("k")
ax_signal.set_title("Smoothing = multiplying spectra")
ax_signal.legend(loc="upper right")
for ax in (ax_plane, ax_gain, ax_signal):
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Compute $\langle e^{2ix}, e^{2ix}\rangle$ and $\langle e^{ix}, e^{-ix}\rangle$.
2. Write $f(x) = 3 + 2\cos 2x - 4\sin x$ as a complex Fourier series (list the nonzero $c_n$).
3. Compute by hand the DFTs of $(1, 0, 0, 0)$ and $(1, 1, 1, 1)$.

### Intermediate

4. **Draw it first, then compute.** Plot the four samples $x = (1, i, -1, -i)$ as points on the unit circle and notice how they rotate from one sample to the next. Which single DFT coefficient do you expect to be nonzero, and what is its value? Then compute the DFT.
5. Compute the Fourier coefficients of the triangle wave $f(x) = |x|$ on $(-\pi, \pi)$ and check the $1/n^2$ decay.
6. A $70$ Hz tone is sampled at $100$ Hz. At what frequency does it appear in the spectrum? What about $130$ Hz? Check with `np.fft`.

### Advanced

7. Apply Parseval's identity to the square wave to show $1 + \frac{1}{9} + \frac{1}{25} + \cdots = \frac{\pi^2}{8}$, and deduce $\sum_{n \ge 1}\frac{1}{n^2} = \frac{\pi^2}{6}$.
8. Prove that for real input $x_k$, $X_{N-n} = \overline{X_n}$. Why is a one-sided spectrum ($0 \le n \le N/2$) enough for real signals, and why is the amplitude estimate $2|X_n|/N$ and not $|X_n|/N$?
9. Show that if $f$ is $2\pi$-periodic and differentiable, then $c_n(f') = in\,c_n(f)$. Explain how this connects to notebooks 01 and 02, where $\frac{d}{dt}$ became multiplication by $i\omega$.

## 7. Solutions

1. $\langle e^{2ix}, e^{2ix}\rangle = \frac{1}{2\pi}\int 1\,dx = 1$. $\langle e^{ix}, e^{-ix}\rangle = \frac{1}{2\pi}\int e^{2ix}dx = 0$.
2. $2\cos 2x = e^{2ix} + e^{-2ix}$ and $-4\sin x = -4\frac{e^{ix} - e^{-ix}}{2i} = 2ie^{ix} - 2ie^{-ix}$. So $c_0 = 3$, $c_{\pm2} = 1$, $c_1 = 2i$, $c_{-1} = -2i$ (conjugates, as they must be for a real $f$).
3. $(1, 0, 0, 0) \mapsto (1, 1, 1, 1)$: an impulse contains all frequencies equally. $(1, 1, 1, 1) \mapsto (4, 0, 0, 0)$: a constant is pure frequency $0$.
4. The samples advance by a quarter turn counter-clockwise each step: $x_k = i^k = \omega^{k}$ with $\omega = e^{2\pi i/4}$. That is basis vector $n = 1$ itself, so only $X_1$ is nonzero and $X_1 = \sum_k \omega^k\omega^{-k} = 4$. Check: $X = (0, 4, 0, 0)$.
5. $c_0 = \frac{1}{2\pi}\int_{-\pi}^{\pi}|x|\,dx = \frac{\pi}{2}$. For $n \ne 0$, $f$ is even so $c_n = \frac{1}{\pi}\int_0^{\pi}x\cos nx\,dx = \frac{1}{\pi}\cdot\frac{(-1)^n - 1}{n^2}$: $-\frac{2}{\pi n^2}$ for odd $n$, $0$ for even. Decay $1/n^2$. ✓
6. $70$ Hz is above the Nyquist frequency $50$ Hz and aliases to $100 - 70 = 30$ Hz. $130$ Hz aliases to $130 - 100 = 30$ Hz as well.
7. $\frac{1}{2\pi}\int|\mathrm{sign}\,x|^2 = 1$, and $\sum|c_n|^2 = 2\sum_{n \text{ odd} > 0}\frac{4}{\pi^2n^2}$. So $\sum_{\text{odd}}\frac{1}{n^2} = \frac{\pi^2}{8}$. Splitting $S = \sum\frac{1}{n^2}$ into odd and even parts, $S = \frac{\pi^2}{8} + \frac{S}{4}$, so $S = \frac{\pi^2}{6}$.
8. $X_{N-n} = \sum_k x_k e^{-2\pi i(N-n)k/N} = \sum_k x_k e^{2\pi i nk/N} = \overline{\sum_k x_k e^{-2\pi i nk/N}} = \overline{X_n}$, using $e^{-2\pi i k} = 1$ and $x_k$ real. The negative-frequency half carries no new information. A real tone $A\cos(\cdot)$ splits its amplitude equally between bins $n$ and $N - n$ (each gets $AN/2$), so to recover $A$ from one side you double: $2|X_n|/N$.
9. Integrate by parts: $c_n(f') = \frac{1}{2\pi}\left[f e^{-inx}\right]_{-\pi}^{\pi} + \frac{in}{2\pi}\int f e^{-inx}dx = 0 + in\,c_n(f)$ (the boundary term vanishes by periodicity). Differentiation is diagonal in the Fourier basis: each mode $e^{inx}$ is an eigenfunction of $\frac{d}{dx}$ with eigenvalue $in$ — the same reason $e^{i\omega t}$ turned ODEs into algebra.

Numerical checks:

In [ ]:
import numpy as np

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

x = np.linspace(-np.pi, np.pi, 200001)
inner = lambda f, g: np.trapezoid(f * np.conj(g), x) / (2 * np.pi)
print("Ex 1:", np.round(inner(np.exp(2j * x), np.exp(2j * x)), 8), np.round(inner(np.exp(1j * x), np.exp(-1j * x)), 8))

f = 3 + 2 * np.cos(2 * x) - 4 * np.sin(x)
print("Ex 2:", {n: np.round(inner(f, np.exp(1j * n * x)), 6) for n in range(-2, 3)})
print("Ex 3:", np.fft.fft([1, 0, 0, 0]), np.fft.fft([1, 1, 1, 1]))
print("Ex 4:", np.round(np.fft.fft([1, 1j, -1, -1j]), 12))
print("Ex 5:", [np.round(inner(np.abs(x), np.exp(1j * n * x)).real, 5) for n in range(0, 4)],
      " formula:", [np.round(np.pi / 2, 5)] + [np.round(((-1) ** n - 1) / (np.pi * n**2), 5) for n in range(1, 4)])

sample_rate = 100
k = np.arange(sample_rate)
for f_tone in (70, 130):
    spectrum = np.abs(np.fft.fft(np.cos(2 * np.pi * f_tone * k / sample_rate)))[: sample_rate // 2 + 1]
    print(f"Ex 6: {f_tone} Hz appears at", np.fft.rfftfreq(sample_rate, 1 / sample_rate)[np.argmax(spectrum)], "Hz")

odd = np.arange(1, 200001, 2)
print("Ex 7:", np.sum(1 / odd**2), "vs pi^2/8 =", np.pi**2 / 8)
signal = np.random.default_rng(0).standard_normal(12)
X = np.fft.fft(signal)
print("Ex 8: X_{N-n} == conj(X_n):", np.allclose(X[1:][::-1], np.conj(X[1:])))
g = np.exp(np.sin(x))
print("Ex 9: c_3(f') =", np.round(inner(np.cos(x) * g, np.exp(3j * x)), 8), " 3i c_3(f) =", np.round(3j * inner(g, np.exp(3j * x)), 8))

## 8. Conclusion

- The exponentials $e^{inx}$ are orthonormal, so a periodic signal's coordinates are $c_n = \frac{1}{2\pi}\int f e^{-inx}dx$, and $f = \sum c_n e^{inx}$ is a chain of arrows turning at integer speeds.
- Real signals have $c_{-n} = \overline{c_n}$; $c_n = \frac{a_n - ib_n}{2}$ links to the sine/cosine form. Parseval's identity is Pythagoras.
- Jumps give $1/n$ decay and the Gibbs overshoot of about $9\%$ of the jump; smoother signals have faster-decaying coefficients.
- Sampling turns the coefficient integral into the **DFT** $X_n = \sum x_k e^{-2\pi i nk/N}$, an orthogonal change of basis built from roots of unity; `np.fft` computes it in $O(N\log N)$. Frequencies are only defined modulo the sampling rate (aliasing), and real signals have mirror-symmetric spectra.
- The **convolution theorem** $\mathrm{DFT}(x * h) = X\cdot H$ makes filtering a multiplication by complex gains $H_n$.

**Next:** notebook 04 builds filters on purpose. The $z$-transform generalises the DFT from the unit circle to the whole complex plane; the poles and zeros of a transfer function $H(z)$ decide whether a recursive filter is stable and which frequencies it passes.